<div align="center">

<pre style="color: #B22222; font-family: 'Courier New', Consolas, monospace; line-height: 1.25;">
███████╗     ██╗████████╗██╗   ██╗    ██╗   ██╗███████╗██╗  ██╗
██╔════╝     ██║╚══██╔══╝██║   ██║    ██║   ██║██╔════╝╚██╗██╔╝
███████╗     ██║   ██║   ██║   ██║    ██║   ██║█████╗   ╚███╔╝ 
╚════██║     ██║   ██║   ██║   ██║    ╚██╗ ██╔╝██╔══╝   ██╔██╗ 
███████║  █████║   ██║   ╚██████╔╝     ╚████╔╝ ███████╗██╔╝ ██╗
  ╚══════╝  ╚════╝   ╚═╝    ╚═════╝       ╚═══╝  ╚══════╝╚═╝  ╚═╝ ©
</pre>

### 机器人实验室发票报销小程序

**一键提取 PDF 发票信息，自动生成 Excel 报表**


---

#### 使用指南

| 步骤 | 操作项 | 详细说明 |
|:---:|:---|:---|
| **1** | **打开终端** | 启动本地命令行工具（Terminal ） |
| **2** | **修改路径** | 将代码中的 `input_dir` 变量替换为你的 **PDF 文件夹绝对路径** |
| **3** | **检查文件夹** | 确认目录内仅包含需处理的 `.pdf` 发票文件，避免混入其他文档 |
| **4** | **运行代码** | 执行主程序，系统自动生成 `test.xlsx` |
| **5** | **找BUG** | 查看哪个发票在作妖 |
| **6** | **输出小程序** | 输出一个可用的小程序 |

##### © KAISEN

</div>


## **1. 终端**

In [ ]:
# pip install pdfplumber
# Conda：conda install -c conda-forge pdfplumber

- **检查环境 - Mac**


In [ ]:
# which python
# which pip
# which pyinstaller
# conda env list #看看有哪些环境
# conda activate xxxx #激活环境
# export PATH="/Library/anaconda3/envs/simple/bin:$PATH" 

## **2. 修改路径**

In [1]:
# 使用前修改文件路径（文件夹: input_dir + 表格位置: output_dir）

input_dir = r"/Users/kaisen/Desktop/25世锦赛" #文件夹位置
output_path = r"/Users/kaisen/Desktop/待报销/小程序/报销.xlsx" #表格位置，运行代码自动覆盖上一个表格

# MAC: r"/Users/kaisen/Desktop/2026"
# WIN: r"C:\Users\kaisen\Desktop\2026"

## **3. 检查文件夹**

In [3]:
import os
from pathlib import Path

try:
    files = os.listdir(input_dir)
    print(f"文件夹内共有 {len(files)} 个项目：")
    for name in files:
        print(f" • {name}")
except Exception as e:
    print(f"[错误] 无法读取 {input_dir}: {e}")

文件夹内共有 37 个项目：
 • _塑料制品_PC板 -160-周智霖.pdf
 • 塑料管 -55-葛宸彤.pdf
 • 3D打印耗材-85-王子豪.pdf
 • 轴承_滚动轴承 -24.2-葛宸彤.pdf
 • .DS_Store
 • ¥64.00+第九队+电动螺丝刀.pdf
 • 气动元件_气缸 -252-葛宸彤.pdf
 • ¥4.50+第九队+橡皮筋.pdf
 • ¥2.59+第九队+美制螺丝.pdf
 • 金属制品铝拉钉 -26.5-葛宸彤.pdf
 • 不锈钢板-450-葛宸彤.pdf
 • ¥14.52+第九队+美制螺丝.pdf
 • _橡胶制品_同步带-50-卢骏驰.pdf
 • ¥24.00+第九队+毛刷.pdf
 • 塑料管 -153-葛宸彤.pdf
 • _黑色金属冶炼压延品_不锈钢板 -340-周智霖.pdf
 • 水晶头-华秋吉-116.68元.pdf
 • 不锈钢板 -90-葛宸彤.pdf
 • 气动元件 - 673 - 葛宸彤.pdf
 • 橡胶制品_同步带 - 195 - 周智霖.pdf
 • _金属制品_铜螺母 - 21.4 - 周智霖.pdf
 • _气动元件_气缸 -248-周智霖.pdf
 • _金属制品_金属制品小孔不锈钢零件 -1000-周智霖.pdf
 • 气动元件_气缸 - 441 - 周智霖.pdf
 • 橡胶制品_同步带 - 190 - 周智霖.pdf
 • 铝材-28-葛宸彤.pdf
 • _橡胶制品_同步带 -70-卢骏驰.pdf
 • 佘屹轩-200-工艺品.pdf
 • 水晶头-华秋吉-39.9元.pdf
 • 40元-第十队-铝方管.pdf
 • 气动元件_气缸 - 207 - 葛宸彤.pdf
 • 3D打印耗材-157.23-董文琦.pdf
 • 董文琦-3D打印耗材-167.60.pdf
 • 90元-第十队-钣金.pdf
 • _金属制品_金属加工 -140-葛宸彤.pdf
 • 橡胶制品_橡皮筋-22-卢骏驰.pdf
 • 董文琦-扎带-24.50.pdf


## **4. 运行代码**

In [ ]:
import fitz  #PyMuPDF
import re
import os
import pandas as pd
from glob import glob



# ========== extract_invoice_data ========== #
#提取信息

def extract_invoice_data(path):
    """从单张PDF电子发票中提取数据，失败返回None"""
    try:
        doc = fitz.open(path)
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        doc.close()
    except Exception as e:
        print(f"⛔️[错误] 无法读取 {path}: {e}")
        return None

    data = {"文件名": os.path.basename(path)}


    # ======== 付款人 ========
    filename = os.path.basename(path)  
    name_without_ext = os.path.splitext(filename)[0] 

    parts = name_without_ext.split('-')
    candidate = parts[-1]  
    data["姓名"] = candidate if candidate else ""
    
    # ======== 发票号码：20位数字 ========
    m = re.search(r"(\d{20})", full_text)
    data["发票号码"] = m.group(1) if m else ""


    # ======== 开票日期 ========
    m = re.search(r"(\d{4}年\d{2}月\d{2}日)", full_text)
    data["开票日期"] = m.group(1) if m else ""


    # ======== 购买方名称 ======== 
    m = re.search(r"(上海交通大学)", full_text)
    data["购买方名称"] = m.group(1) if m else ""


    # ======== 供应商 ======== #
    """ ======== 待优化 ======== """
    try:
        pairs = re.findall(r"([^\n]{2,30}?)\s*\n\s*([A-Z0-9]{15,20})", full_text)

        valid_pairs = [(name, tax) for name, tax in pairs 
                if re.search(r'[A-Z]', tax) and re.search(r'\d', tax) or re.search(r'\d{18}', tax) ]

        data["供应商"] = valid_pairs[2][0] if len(valid_pairs) >= 3 else valid_pairs[1][0]
    except Exception as e:
        print(f'   ⛔️ {path} 无供应商')
        data["供应商"] = "⛔️"


    # ======== 数量 + 单位 ======== #
    # 虽然代码识别不同发票的差异大，但数量一般集中在单位后四行内
    # 先搜索单位后，遍历单位后四行的数值，只搜寻整数
    # 输出整数的同时输出相应的单位
    """ ======== 优化空间 ======== """
    # 单位的识别
    # 搜索方式

    lines = [t.strip() for t in re.split(r'\s+', full_text) if t.strip()]
    units = ["条", "套", "个", "盒", "卷", "支", "包", "批", "台", "件",
             "根", "只", "把", "米", "次", "份", "粒", "克", "斤"]

    quantity = 0
    for i, line in enumerate(lines):
        if line in units:
            next_line_i = i+1
            for i in range (next_line_i,next_line_i+4):
                next_line = lines[i]
                if re.match(r"^\d+$", next_line):  # 纯整数
                    quantity += int(next_line)
                    data["数量"] = quantity


    # ======== 金额：提取所有 ¥开头的金额 ========
    #查找以¥开头的数字，取最大值

    amounts = re.findall(r"¥\s*(\d+\.\d{2})", full_text)
    data["金额"] = max(amounts) if amounts else ""


    # ======== 单价 ======== 
    # 金额 / 数量

    try:
        ave = round(float(max(amounts)) / int(quantity),2)
    except Exception as e:
        print(f'   ⛔️ {path} 无单价')
        ave = 0
    data["单价"] = ave


    # ======== 类型：易耗品 + 低值判断 ======== 
    # 通过单价判断
    
    data["类型"] = "易耗品" if ave < 200 else "❗️低值"


    # ======== 项目名称  ========
    # 文件名用 “-” 分割，取倒一

    try:
        m = re.findall(r"\*([\u4e00-\u9fa5A-Z0-9]+)\*", full_text)
        data["项目名称"] = m[0]

    except Exception as e:
        print(f'   ⛔️ {path} 无项目名称')
        data["项目名称"] = "⛔️"


    # ======== 规格型号 ========
    # 文件名用 “-” 分割，取倒一

    try:
        a = re.findall(r"\*[\u4e00-\u9fa5A-Z0-9]+\*[\w\u4e00-\u9fa5A-Z0-9]+\n([\w\u4e00-\u9fa5A-Z0-9]+)", full_text)
        candidate = ""
        for can in a:
            candidate += "\n" + can
        if len(candidate) > 4:
            data["规格型号"] = candidate
        else:
            data["规格型号"] = ""
        # ["1","2","3","4","5","6","7","8","9","0"]
    except Exception as e:
        print(f'   ⛔️ {path} 规格型号')
        data["项目名称"] = "⛔️"



    # ======== 单位 ======== 
    m = re.search(r"([套件卷支个包盒条批份]+)", full_text)
    unit = m.group(1) if m else ""
    unit += "❌" if unit in r"套包盒批份 " else ""
    data["单位"] = unit

    return data



# ========== batch_invoices_to_excel ========== #
#遍历文件 + 提取信息

def batch_invoices_to_excel(input_folder, output_excel_path):
    """
    遍历文件夹内所有PDF，提取发票信息并汇总到Excel。
    """
    pdf_files = sorted(glob(os.path.join(input_folder, "*.pdf")))
    if not pdf_files:
        print(f"❗️文件夹 {input_folder} 内未找到PDF文件❗️")
        return

    print(f"✅ 共找到 {len(pdf_files)} 个PDF文件，开始处理...😵‍💫😵‍💫😵‍💫")

    records = []
    for pdf_path in pdf_files:
        print(f"   • 正在处理: {os.path.basename(pdf_path)}")
        record = extract_invoice_data(pdf_path)
        if record:
            records.append(record)

    if not records:
        print("[错误] 未能从任何PDF中提取到有效数据。")
        return

    # 定义列顺序：同一种data分到一列
    columns = [
        "文件名", "类型", "项目名称", "单价", "数量", 
        "单位", "金额", "发票号码", 
        "开票日期", "供应商","规格型号"
    ]

    df = pd.DataFrame(records)
    # 确保列顺序统一，缺失列补空字符串
    for col in columns:
        if col not in df.columns:
            df[col] = ""
    df = df[columns]

    # 生成Excel
    df.to_excel(output_excel_path, index=False, engine="openpyxl")
    print(f"✅ Excel已保存至: {output_excel_path}")
    print(f"✅ 成功处理 {len(records)} 张发票 \n🫠 为您节省了{round(len(records)*0.55,2)}分钟")
    return df


# ================ MAIN ================ #

if __name__ == "__main__":
    input_dir
    output_path
    
    batch_invoices_to_excel(input_dir, output_path)

✅ 共找到 36 个PDF文件，开始处理...😵‍💫😵‍💫😵‍💫
   • 正在处理: 3D打印耗材-157.23-董文琦.pdf
   • 正在处理: 3D打印耗材-85-王子豪.pdf
   • 正在处理: 40元-第十队-铝方管.pdf
   • 正在处理: 90元-第十队-钣金.pdf
   ⛔️ /Users/kaisen/Desktop/25世锦赛/90元-第十队-钣金.pdf 无单价
   • 正在处理: _塑料制品_PC板 -160-周智霖.pdf
   • 正在处理: _橡胶制品_同步带 -70-卢骏驰.pdf
   • 正在处理: _橡胶制品_同步带-50-卢骏驰.pdf
   • 正在处理: _气动元件_气缸 -248-周智霖.pdf
   • 正在处理: _金属制品_金属制品小孔不锈钢零件 -1000-周智霖.pdf
   • 正在处理: _金属制品_金属加工 -140-葛宸彤.pdf
   • 正在处理: _金属制品_铜螺母 - 21.4 - 周智霖.pdf
   • 正在处理: _黑色金属冶炼压延品_不锈钢板 -340-周智霖.pdf
   • 正在处理: ¥14.52+第九队+美制螺丝.pdf
   • 正在处理: ¥2.59+第九队+美制螺丝.pdf
   • 正在处理: ¥24.00+第九队+毛刷.pdf
   ⛔️ /Users/kaisen/Desktop/25世锦赛/¥24.00+第九队+毛刷.pdf 无供应商
   ⛔️ /Users/kaisen/Desktop/25世锦赛/¥24.00+第九队+毛刷.pdf 无单价
   ⛔️ /Users/kaisen/Desktop/25世锦赛/¥24.00+第九队+毛刷.pdf 无项目名称
   • 正在处理: ¥4.50+第九队+橡皮筋.pdf
   • 正在处理: ¥64.00+第九队+电动螺丝刀.pdf
   • 正在处理: 不锈钢板 -90-葛宸彤.pdf
   • 正在处理: 不锈钢板-450-葛宸彤.pdf
   • 正在处理: 佘屹轩-200-工艺品.pdf
   • 正在处理: 塑料管 -153-葛宸彤.pdf
   • 正在处理: 塑料管 -55-葛宸彤.pdf
   • 正在处理: 橡胶制品_同步带 - 190 - 周智霖.pdf
   • 正在处理: 橡胶制品

# **5. PDF文本（查BUG）**

**原理：通过定位PDF文本中的特定规律获取期望内容**

In [14]:
def extract_invoice_data(path):
    """从单张PDF电子发票中提取结构化数据，失败时返回None"""
    try:
        doc = fitz.open(path)
        full_text = ""
        for page in doc:
            full_text += page.get_text()
        print(full_text)
        doc.close()
    except Exception as e:
        print(f"[错误] 无法读取 {path}: {e}")
        return None

    data = {"文件名": os.path.basename(path)}

#这里测试发票内容抓取方式
    # ==================== 付款人 ====================
    filename = os.path.basename(path)  
    name_without_ext = os.path.splitext(filename)[0] 
    print(name_without_ext)

    parts = name_without_ext.split('-')
    print(parts)
    candidate = parts[-1]  
    print(candidate)
    data["姓名"] = candidate if candidate else ""
    
    #if re.match(r"^[\u4e00-\u9fa5]{2,4}", candidate):
    #    data["姓名"] = candidate
    #else:
    #    data["姓名"] = ""

    # ======== 供应商 ========
    try:
        pairs = re.findall(r"([^\n]{2,30}?)\s*\n\s*([A-Z0-9]{15,20})", full_text)

        valid_pairs = [(name, tax) for name, tax in pairs 
                if re.search(r'[A-Z]', tax) and re.search(r'\d', tax) or re.search(r'\d{18}', tax) ]

        data["供应商"] = valid_pairs[2][0] if len(valid_pairs) >= 3 else valid_pairs[1][0]
    except Exception as e:
        print(e)
        data["供应商"] = "转人工❌"


    # ======== 数量 ========
    lines = [t.strip() for t in re.split(r'\s+', full_text) if t.strip()]
    units = ["条", "套", "个", "盒", "卷", "支", "包", "批", "台", "件",
             "根", "只", "把", "米", "次", "份", "粒", "克", "斤"]

    quantity = 0
    for i, line in enumerate(lines):
        if line in units:
            print('line',line)
            next_line_i = i+1
            for i in range (next_line_i,next_line_i+4):
                next_line = lines[i]
                if re.match(r"^\d+$", next_line):  # 纯整数
                    quantity += int(next_line)
                    print('q',quantity)
                    data["数量"] = quantity
                    


    # ======== 金额：提取所有 ¥开头的金额 ========
    amounts = re.findall(r"¥\s*(\d+\.\d{2})", full_text)
    data["金额"] = max(amounts) if amounts else ""
    

def batch_invoices_to_excel(input_folder, output_excel_path):
    """
    遍历文件夹内所有PDF，提取发票信息并汇总到Excel。
    每行 = 一张发票，每列 = 一个字段（同一种data分到同一列）。
    """

#在这里输入发票路径
    pdf_files = ["/Users/kaisen/Desktop/25世锦赛/_金属制品_金属加工 -140-葛宸彤.pdf"] #在这里测试搞事情的发票！！！
    if not pdf_files:
        print(f"[提示] 文件夹 {input_folder} 内未找到PDF文件。")
        return

    print(f"[信息] 共找到 {len(pdf_files)} 个PDF文件，开始处理...")

    records = []
    for pdf_path in pdf_files:
        print(f"  正在处理: {os.path.basename(pdf_path)}")
        record = extract_invoice_data(pdf_path)
        if record:
            records.append(record)

if __name__ == "__main__":
    input_dir = r"/Users/kaisen/Desktop/25世锦赛"   # 改成你的PDF文件夹路径
    output_path = r"/Users/kaisen/Desktop/test.xlsx"
    
    batch_invoices_to_excel(input_dir, output_path)

[信息] 共找到 1 个PDF文件，开始处理...
  正在处理: _金属制品_金属加工 -140-葛宸彤.pdf
电子发票（普通发票）
发票号码：
开票日期：
购
买
方
信
息
统一社会信用代码/纳税人识别号：
销
售
方
信
息
统一社会信用代码/纳税人识别号：
名称：
名称：
项目名称
规格型号
单 位
数 量
单 价
金 额
税率/征收率
税 额
合
计
价税合计（大写）
（小写）
备
注
开票人：
24322000000366891126
2024年09月26日
上海交通大学
1210000042500615X0
昆山市千灯镇新语睿精密五金加工厂
92320583MAC1BDJ22F
¥138.61
¥1.39
壹佰肆拾圆整
¥140.00
鲁立霞
*金属制品*金属加工
1%
套
138.61
1.39
138.613861386139
1
下载次数：1   

_金属制品_金属加工 -140-葛宸彤
['_金属制品_金属加工 ', '140', '葛宸彤']
葛宸彤
line 套
q 1


# **6. 输出小程序**

- **终端**


In [ ]:
# pip install pyinstaller

- **MAC：输出.app**

In [ ]:
# pyinstaller --windowed --name "xxxx" invoice_gui.py

- **WIN：输出.exe**


In [ ]:
# pyinstaller --onefile --windowed --name "xxxx" invoice_gui.py

- **Git**

In [ ]:
# git push origin main 